In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split
#from imblearn.over_sampling import RandomOverSampler
from interpret.glassbox import (LogisticRegression,
                                ClassificationTree, 
                                ExplainableBoostingClassifier)
from interpret import show
from sklearn.metrics import f1_score, accuracy_score

D:\Users\cccc\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [5]:
pip install interpret

  Using cached interpret-0.7.8-py3-none-any.whl.metadata (1.2 kB)
  Using cached interpret_core-0.7.8-py3-none-any.whl.metadata (2.9 kB)
  Using cached salib-1.5.2-py3-none-any.whl.metadata (11 kB)
  Using cached shap-0.52.0-cp312-abi3-win_amd64.whl.metadata (26 kB)
  Using cached aplr-10.26.0-cp313-cp313-win_amd64.whl.metadata (1.2 kB)
  Using cached dash-4.4.1-py3-none-any.whl.metadata (11 kB)
  Using cached dash_cytoscape-1.0.2.tar.gz (4.0 MB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached gevent-26.8.0-cp313-cp313-win_amd64.whl.metadata (14 kB)
  Using cached retrying-1.4.2-py3-none-any.whl.metadata (5.5 kB)
  Using cached janus-2.0.0-py3-none-any.whl.metadata (5.3 kB)
  Using cached greenlet-3.5.5-cp313-cp313-win_amd64.whl.metadata (3.9 kB)
  Using cached cffi-2.1.1-cp313-cp313-win_amd64.whl.metadata (2.6 kB)
  Using cached zope_event-6.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached slicer-0.0.8-py3-none-an

  DEPRECATION: Building 'dash-cytoscape' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'dash-cytoscape'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fg-data-profiling 4.19.1 requires matplotlib<=3.10,>=3.5, but you have matplotlib 3.11.1 which is incompatible.
fg-data-profiling 4.19.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.5 which is incompatible.
fg-data-profiling 4.19.1 requires scipy<1.17,>=1.8, but you

In [2]:
data = pd.read_csv("Datasets/healthcare-dataset-stroke-data.csv")

In [3]:
# One-hot encode all categorical columns
categorical_cols = ["gender","ever_married","work_type","Residence_type","smoking_status"]
encoded = pd.get_dummies(data[categorical_cols], prefix=categorical_cols)

In [4]:
# Update data with new columns
data = pd.concat([encoded, data], axis=1)
data.drop(categorical_cols, axis=1, inplace=True)

In [5]:
# Impute missing values of BMI
data.bmi = data.bmi.fillna(0)

In [6]:
# Drop id as it is not relevant
data.drop(["id"], axis=1, inplace=True)

In [7]:
X = data.iloc[:,:-1]
y = data.iloc[:,-1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=2021)

In [8]:
print(X_train.shape)
print(X_test.shape)

(4088, 21)
(1022, 21)


In [9]:
# %% Fit logistic regression model
lr = LogisticRegression(random_state=2021, feature_names=X_train.columns, penalty='l1', solver='liblinear')
lr.fit(X_train, y_train)
print("Training finished.")

D:\Users\cccc\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
D:\Users\cccc\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Training finished.


In [10]:
# %% Evaluate logistic regression model
y_pred = lr.predict(X_test)
print(f"F1 Score {f1_score(y_test, y_pred, average='macro')}")
print(f"Accuracy {accuracy_score(y_test, y_pred)}")

F1 Score 0.5048585401526577
Accuracy 0.9481409001956947


In [14]:
# %% Explain local prediction
lr_local = lr.explain_local(X_test[:100], y_test[:100], name='Logistic Regression')
show(lr_local)

<!-- http://127.0.0.1:7001/1774233708624/ -->

In [12]:
# %% Explain global logistic regression model
lr_global = lr.explain_global(name='Logistic Regression')
show(lr_global)

<!-- http://127.0.0.1:7001/2676322336848/ -->

In [13]:
# %% Fit decision tree model
tree = ClassificationTree()
tree.fit(X_train, y_train)
print("Training finished.")
y_pred = tree.predict(X_test)
print(f"F1 Score {f1_score(y_test, y_pred, average='macro')}")
print(f"Accuracy {accuracy_score(y_test, y_pred)}")

Training finished.
F1 Score 0.5073148175476915
Accuracy 0.952054794520548


In [14]:
# %% Explain local prediction
tree_local = tree.explain_local(X_test[:100], y_test[:100], name='Tree')
show(tree_local)

<!-- http://127.0.0.1:7001/2676336813664/ -->

In [15]:
# %% Fit Explainable Boosting Machine
ebm = ExplainableBoostingClassifier(random_state=2021)
ebm.fit(X_train, y_train) 
print("Training finished.")
y_pred = ebm.predict(X_test)
print(f"F1 Score {f1_score(y_test, y_pred, average='macro')}")
print(f"Accuracy {accuracy_score(y_test, y_pred)}")

Training finished.
F1 Score 0.5235431235431235
Accuracy 0.9500978473581213


In [16]:
# %% Explain locally
ebm_local = ebm.explain_local(X_test[:100], y_test[:100], name='EBM')
show(ebm_local)

<!-- http://127.0.0.1:7001/2676336810640/ -->